In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
data_dir = "/content/drive/MyDrive"

Mounted at /content/drive


In [2]:
import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler

import matplotlib.pyplot as plt

from PIL import Image
import torchvision
from torchvision import transforms, datasets

from torch.utils.data import (
    DataLoader,
    random_split
)

In [3]:
import shutil

if not os.path.exists("/content/dataset_split"):

    shutil.copytree(
        "/content/drive/MyDrive/dataset_split",
        "/content/dataset_split"
    )

print("Dataset skopiowany lokalnie")

Dataset skopiowany lokalnie


In [11]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])



train_dir = "/content/dataset_split/train"
val_dir = "/content/dataset_split/val"
test_dir = "/content/dataset_split/test"

train_dataset = datasets.ImageFolder(
    root=train_dir,
    transform=transform
)

val_dataset = datasets.ImageFolder(
    root=val_dir,
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root=test_dir,
    transform=transform
)

# dataloadery
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    persistent_workers=True
)

# klasy
class_names = train_dataset.classes

print("Klasy:")
print(class_names)

print(f"\nTrain images: {len(train_dataset)}")
print(f"Val images: {len(val_dataset)}")
print(f"Test images: {len(test_dataset)}")

Klasy:
['APSK128', 'APSK16', 'APSK32', 'APSK64', 'ASK4', 'ASK8', 'BPSK', 'GMSK', 'PSK16', 'PSK32', 'PSK8', 'QAM128', 'QAM16', 'QAM256', 'QAM32', 'QAM64', 'QPSK']

Train images: 18370
Val images: 3923
Test images: 3923


# **EfficientNet-B0**

In [12]:
from torchvision.models import ( efficientnet_b0, EfficientNet_B0_Weights )

model = efficientnet_b0( weights=EfficientNet_B0_Weights.IMAGENET1K_V1 )

model.classifier[1] = nn.Linear( model.classifier[1].in_features, len(train_dataset.classes))
model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.classifier[1].in_features, 256),
    nn.ReLU(),
    nn.Linear(256, len(train_dataset.classes))
)

In [13]:
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True


batch_size = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.classifier.parameters(),lr=5e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2
)

In [14]:
num_epochs = 25
best_val_accuracy = 0.0

train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    if epoch == 5:

        print("Odblokowuję features")

        for param in model.features[6:].parameters():
            param.requires_grad = True

        optimizer = optim.Adam([
            {'params': model.features[6:].parameters(), 'lr': 1e-5},
            {'params': model.classifier.parameters(),     'lr': 5e-5}
        ])

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='max',
            factor=0.5,
            patience=2
        )
    if epoch == 15:

        print("Stage 3")

        for param in model.features.parameters():
            param.requires_grad = True

        optimizer = optim.Adam([
            {'params': model.features.parameters(), 'lr': 2e-6},
            {'params': model.classifier.parameters(), 'lr': 2e-5}
        ])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='max',
            factor=0.5,
            patience=2
        )

    print(f"\nEpoka {epoch+1}/{num_epochs}")



    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_accuracy = correct / total

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)



    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_loss = val_running_loss / val_total
    val_accuracy = val_correct / val_total

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)



    print(
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_accuracy:.4f}"
    )

    print(
        f"Val loss:   {val_loss:.4f} | "
        f"Val acc:   {val_accuracy:.4f}"
    )



    torch.save(
        model.state_dict(),
        "/content/drive/MyDrive/runs_efficientNetB0_1/model_current_efficientNetB0.pth"
    )

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            "/content/drive/MyDrive/runs_efficientNetB0_1/model_best_efficientNetB0.pth"
        )

        print("Zapisano najlepszy model")

    scheduler.step(val_accuracy)

print(f"Best accuracy: {best_val_accuracy:.4f}")


Epoka 1/25
Train loss: 2.5859 | Train acc: 0.3097
Val loss:   2.3511 | Val acc:   0.3622
Zapisano najlepszy model

Epoka 2/25
Train loss: 1.9787 | Train acc: 0.5076
Val loss:   1.9052 | Val acc:   0.4418
Zapisano najlepszy model

Epoka 3/25
Train loss: 1.5771 | Train acc: 0.5924
Val loss:   1.6286 | Val acc:   0.5032
Zapisano najlepszy model

Epoka 4/25
Train loss: 1.3416 | Train acc: 0.6412
Val loss:   1.4734 | Val acc:   0.5468
Zapisano najlepszy model

Epoka 5/25
Train loss: 1.1973 | Train acc: 0.6711
Val loss:   1.3703 | Val acc:   0.5605
Zapisano najlepszy model
Odblokowuję features

Epoka 6/25
Train loss: 0.9525 | Train acc: 0.7187
Val loss:   0.9739 | Val acc:   0.6834
Zapisano najlepszy model

Epoka 7/25
Train loss: 0.7388 | Train acc: 0.7690
Val loss:   0.7829 | Val acc:   0.7204
Zapisano najlepszy model

Epoka 8/25
Train loss: 0.6181 | Train acc: 0.7934
Val loss:   0.6632 | Val acc:   0.7645
Zapisano najlepszy model

Epoka 9/25
Train loss: 0.5389 | Train acc: 0.8217
Val loss

In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/runs_efficientNetB0_1/model_best_efficientNetB0.pth"
    )
)

model.eval()

true_labels = []
pred_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        true_labels.extend(labels.cpu().numpy())
        pred_labels.extend(predicted.cpu().numpy())

In [16]:
runs_dir = "/content/drive/MyDrive/runs_efficientNetB0_1"
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(true_labels, pred_labels)

precision = precision_score(
    true_labels,
    pred_labels,
    average='macro'
)

recall = recall_score(
    true_labels,
    pred_labels,
    average='macro'
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average='macro'
)

print("\nWyniki modelu")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

metrics_file = os.path.join(
    runs_dir,
    "test_metrics.txt"
)

with open(metrics_file, "w") as f:

    f.write("Wyniki modelu\n")
    f.write(f"Accuracy : {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall   : {recall:.4f}\n")
    f.write(f"F1-score : {f1:.4f}\n")

print(f"\nZapisano do: {metrics_file}")


Wyniki modelu
Accuracy : 0.8580
Precision: 0.8660
Recall   : 0.8551
F1-score : 0.8570

Zapisano do: /content/drive/MyDrive/runs_efficientNetB0_1/test_metrics.txt


In [17]:
import seaborn as sns

from sklearn.metrics import confusion_matrix
runs_dir = "/content/drive/MyDrive/runs_efficientNetB0_1"

os.makedirs(runs_dir, exist_ok=True)

# loss plot

plt.figure(figsize=(10,5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')

plt.title("Loss")
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.legend()

plt.savefig(
    os.path.join(runs_dir, "loss_plot.png"),
    bbox_inches='tight'
)

plt.close()

# accuracy plot

plt.figure(figsize=(10,5))

plt.plot(train_accuracies, label='Train Accuracy')
plt.plot(val_accuracies, label='Val Accuracy')

plt.title("Accuracy")
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.legend()

plt.savefig(
    os.path.join(runs_dir, "accuracy_plot.png"),
    bbox_inches='tight'
)

plt.close()

import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(
    true_labels,
    pred_labels
)

plt.figure(figsize=(14,12))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=test_dataset.classes,
    yticklabels=test_dataset.classes
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.savefig(
    os.path.join(
        runs_dir,
        "test_confusion_matrix.png"
    ),
    bbox_inches="tight"
)

plt.close()

print("\nZapisano:")
print("- model_best.pth")
print("- model_current.pth")
print("- loss_plot.png")
print("- accuracy_plot.png")
print("- confusion_matrix.png")


Zapisano:
- model_best.pth
- model_current.pth
- loss_plot.png
- accuracy_plot.png
- confusion_matrix.png
